# Data Preparation & Feature Transformation

This notebook validates data quality and engineers defect-related features before modeling.


In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path

root = Path.cwd()
suppliers = pd.read_csv(root / 'data' / 'suppliers.csv')
batches = pd.read_csv(root / 'data' / 'component_batches.csv')
qc = pd.read_csv(root / 'data' / 'qc_results.csv')
defects = pd.read_csv(root / 'data' / 'defects.csv')

# Missing value summary
print('Suppliers missing values:
', suppliers.isna().sum())
print('Batches missing values:
', batches.isna().sum())
print('QC missing values:
', qc.isna().sum())
print('Defects missing values:
', defects.isna().sum())

# Duplicate detection
print('Duplicate batch ids:', batches['batch_id'].duplicated().sum())
print('Duplicate unit ids:', qc['unit_id'].duplicated().sum())

# Batch size outlier detection (z-score)
mean_batch = batches['quantity_produced'].mean()
std_batch = batches['quantity_produced'].std(ddof=0)
zscores = (batches['quantity_produced'] - mean_batch) / std_batch
outliers = batches.loc[np.abs(zscores) > 3, ['batch_id', 'quantity_produced']]
print('Outlier batch count:', len(outliers))
print(outliers.head())

# Type validation
print(batches.dtypes)
print(qc.dtypes)
print(defects.dtypes)


In [ ]:
# Engineer defect metrics
failed_units = qc[qc['result'] == 'Fail'].copy()
failed_count = failed_units.groupby('batch_id').size().reset_index(name='failed_units')

# defective batch counts from defects data
defect_count = defects.groupby('batch_id').size().reset_index(name='defect_count')

merged = batches.merge(suppliers, on='supplier_id', how='left')
merged = merged.merge(failed_count, on='batch_id', how='left')
merged = merged.merge(defect_count, on='batch_id', how='left')

merged['failed_units'] = merged['failed_units'].fillna(0).astype(int)
merged['defect_count'] = merged['defect_count'].fillna(0).astype(int)
merged['defect_rate'] = merged['defect_count'] / merged['quantity_produced']
merged['batch_size'] = merged['quantity_produced']
merged['quality_pass_rate'] = merged['quantity_passed_qc'] / merged['quantity_produced']

q75 = merged['defect_rate'].quantile(0.75)
merged['binary_target'] = (merged['defect_rate'] > q75).astype(int)

# Supplier metrics
supplier_metrics = merged.groupby('supplier_id').agg(
    mean_defect_rate=('defect_rate', 'mean'),
    std_defect_rate=('defect_rate', 'std'),
    num_batches=('batch_id', 'count'),
    total_units=('quantity_produced', 'sum'),
).reset_index()
merged = merged.merge(supplier_metrics, on='supplier_id', how='left')

# Temporal features
merged['production_date'] = pd.to_datetime(merged['production_date'])
merged['received_date'] = pd.to_datetime(merged['received_date'])
merged['days_since_start'] = (merged['production_date'] - merged['production_date'].min()).dt.days
merged['month_of_year'] = merged['production_date'].dt.month
merged['quarter'] = merged['production_date'].dt.quarter
merged['batch_age_days'] = (pd.Timestamp.today().normalize() - merged['production_date']).dt.days

# Component defect rates
component_rate = merged.groupby('component_name')['defect_rate'].mean().reset_index().rename(columns={'defect_rate': 'component_defect_rate'})
merged = merged.merge(component_rate, on='component_name', how='left')

# Fill missing values
merged['defect_count'] = merged['defect_count'].fillna(0)
merged['mean_defect_rate'] = merged['mean_defect_rate'].fillna(0)
merged['std_defect_rate'] = merged['std_defect_rate'].fillna(0)
merged['component_defect_rate'] = merged['component_defect_rate'].fillna(0)

merged.head()


In [ ]:
# Save analytical dataset for downstream modeling
output_path = root / 'data' / 'analytical_dataset.csv'
merged.to_csv(output_path, index=False)
print(f'Exported cleaned dataset to {output_path}')
